In [1]:
import pandas as pd
import numpy as np
import kagglehub
import os

/opt/homebrew/Caskroom/miniforge/base/envs/houseprice/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = kagglehub.dataset_download(
    "shashanknecrothapa/ames-housing-dataset"
)

df = pd.read_csv(path + "/AmesHousing.csv")

df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [3]:
df_fe = df.copy()

In [4]:
df_fe["House Age"] = 2026 - df_fe["Year Built"]

df_fe["Remodeled"] = (
    df_fe["Year Remod/Add"] != df_fe["Year Built"]
).astype(int)

df_fe["Has Pool"] = df_fe["Pool QC"].notnull().astype(int)

df_fe["Has Garage"] = df_fe["Garage Type"].notnull().astype(int)

df_fe["Has Fireplace"] = df_fe["Fireplace Qu"].notnull().astype(int)

In [5]:
df_fe["Total Area"] = (
    df_fe["1st Flr SF"]
    + df_fe["2nd Flr SF"]
    + df_fe["Total Bsmt SF"]
)

df_fe["Total Bath"] = (
    df_fe["Full Bath"]
    + 0.5 * df_fe["Half Bath"]
    + df_fe["Bsmt Full Bath"]
    + 0.5 * df_fe["Bsmt Half Bath"]
)

df_fe["Quality_Area"] = (
    df_fe["Overall Qual"] * df_fe["Gr Liv Area"]
)

In [6]:
new_features = [
    "House Age",
    "Remodeled",
    "Has Pool",
    "Has Garage",
    "Has Fireplace",
    "Total Area",
    "Total Bath",
    "Quality_Area"
]

df_fe[new_features].head()

,House Age,Remodeled,Has Pool,Has Garage,Has Fireplace,Total Area,Total Bath,Quality_Area
0,66,0,0,1,1,2736.0,2.0,9936
1,65,0,0,1,0,1778.0,1.0,4480
2,68,0,0,1,0,2658.0,1.5,7974
3,58,0,0,1,1,4220.0,3.5,14770
4,29,1,0,1,1,2557.0,2.5,8145


In [7]:
df_fe[new_features + ["SalePrice"]].corr()["SalePrice"].sort_values(
    ascending=False
)

SalePrice        1.000000
Quality_Area     0.845441
Total Area       0.793024
Total Bath       0.635694
Has Fireplace    0.481446
Has Garage       0.225950
Has Pool         0.087960
Remodeled       -0.047274
House Age       -0.558426
Name: SalePrice, dtype: float64

In [8]:
df_fe.to_csv("../data/feature_engineered.csv", index=False)